# Final private inference — shard 0 of 2 (A100)

Processes original rows 0–999 with the exact pinned deterministic adaptive policy. Run this notebook only in session A. Session B must use the shard-1 notebook and a different output directory.

In [ ]:
# Cell 1 — Clone the pinned inference code and install. Restart once afterward.
import subprocess,sys
from pathlib import Path
CODE_COMMIT="24380f2ab9f4a4d7af417193b44b2fa0da22b7ed"
REPO=Path("/content/qwen-math-final-2026")
if not REPO.exists(): subprocess.run(["git","clone","https://github.com/jhparktime/qwen-math-final-2026.git",str(REPO)],check=True)
subprocess.run(["git","-C",str(REPO),"fetch","origin",CODE_COMMIT],check=True)
subprocess.run(["git","-C",str(REPO),"checkout","--detach",CODE_COMMIT],check=True)
actual=subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip();assert actual==CODE_COMMIT
subprocess.run([sys.executable,"-m","pip","install","-q","--no-cache-dir","-r",str(REPO/"requirements-colab.txt")],check=True)
subprocess.run([sys.executable,"-m","pip","uninstall","-q","-y","torchcodec"],check=False)
print("[CODE COMMIT]",actual);print("[SETUP] restart runtime once, then continue at Cell 2")

In [ ]:
# Cell 2 — Restore state after restart and mount Drive.
import subprocess
from pathlib import Path
from google.colab import drive
REPO=Path("/content/qwen-math-final-2026");CODE_COMMIT="24380f2ab9f4a4d7af417193b44b2fa0da22b7ed"
actual=subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip();assert actual==CODE_COMMIT
drive.mount("/content/drive")
print("[CODE COMMIT]",actual)

In [ ]:
# Cell 3 — One-time base-model cache preparation. No test row is read or sent.
import os
from huggingface_hub import snapshot_download
os.environ.pop("HF_HUB_OFFLINE",None);os.environ.pop("TRANSFORMERS_OFFLINE",None)
model_cache=snapshot_download(repo_id="Qwen/Qwen2.5-3B-Instruct",revision="aa8e72537993ba99e69dfaafa59ed015b17504d1")
print("[BASE MODEL CACHED]",model_cache)

In [ ]:
# Cell 4 — Deterministically create shard 0 (first 1,000 rows) and its local config.
import json,pandas as pd
SHARD_INDEX=0;NUM_SHARDS=2;EXPECTED_FULL_ROWS=2000
FULL_INPUT=Path("/content/drive/MyDrive/test_submission.csv")
ADAPTER_PATH=Path("/content/drive/MyDrive/2026소중한챌린지/runs/RFT-0008D-r3mix-r2continue-r16-a100/adapter_final")
RUN_ROOT=Path("/content/drive/MyDrive/2026소중한챌린지/runs/FINAL-0006-r3-adaptive-pal3-deterministic-2shard")
OUTPUT_DIR=RUN_ROOT/"shard_0_of_2";OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
full=pd.read_csv(FULL_INPUT,dtype=str,keep_default_na=False);full.columns=[str(c).lstrip("\ufeff").strip() for c in full.columns]
assert len(full)==EXPECTED_FULL_ROWS and {"id","question"}.issubset(full.columns) and full.id.is_unique
assert "answer" not in full.columns or not (full.answer.str.strip()!="").any()
start=SHARD_INDEX*EXPECTED_FULL_ROWS//NUM_SHARDS;end=(SHARD_INDEX+1)*EXPECTED_FULL_ROWS//NUM_SHARDS
shard=full.iloc[start:end][["id","question"]].copy();assert len(shard)==1000
INPUT_PATH=OUTPUT_DIR/"input_shard_0_of_2.csv";shard.to_csv(INPUT_PATH,index=False,encoding="utf-8")
config=json.loads((REPO/"configs/final_inference.json").read_text());config["expected_rows"]=len(shard);config["run_id"]+="-shard0of2"
CONFIG_PATH=OUTPUT_DIR/"config_shard_0_of_2.json";CONFIG_PATH.write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding="utf-8")
assert (ADAPTER_PATH/"adapter_config.json").exists();print("[SHARD]",start,end,len(shard));print(INPUT_PATH,OUTPUT_DIR,sep="\n")

In [ ]:
# Cell 5 — Offline, resume-safe shard inference with live logs.
import os,subprocess,sys
env=os.environ.copy();env["PYTHONPATH"]=str(REPO);env["HF_HUB_OFFLINE"]="1";env["TRANSFORMERS_OFFLINE"]="1"
command=[sys.executable,str(REPO/"inference/final_inference.py"),"--input",str(INPUT_PATH),"--adapter",str(ADAPTER_PATH),"--output-dir",str(OUTPUT_DIR),"--config",str(CONFIG_PATH)]
process=subprocess.Popen(command,cwd=REPO,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,bufsize=1)
for line in process.stdout: print(line,end="",flush=True)
exit_code=process.wait()
if exit_code: raise RuntimeError(f"Inference failed with exit code {exit_code}")

In [ ]:
# Cell 6 — Validate exactly 1,000 shard rows.
SUBMISSION=OUTPUT_DIR/"submissions/submission.csv"
subprocess.run([sys.executable,str(REPO/"scripts/validate_submission.py"),"--input",str(INPUT_PATH),"--submission",str(SUBMISSION),"--expected-rows","1000"],cwd=REPO,check=True)
print("[SHARD 0 COMPLETE]",SUBMISSION)